# Two-Head Attention MGRPO

This notebook uses exactly two custom attention heads:

1. **Generation attention head**: always used before candidate generation. It attends to the classifier probability distribution and chooses the `N` modality/profile candidates to sample.
2. **Preference update attention head**: always used before the LoRA update. It attends to the candidate responses using user-preference reward features and decides which candidates drive the policy update.

The user prompt itself is not rewritten and no modality text is injected into the prompt.


In [1]:
%pip install -q datasets transformers accelerate peft


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Imports and Configuration

Keep your Hugging Face token in `HF_TOKEN` or `HUGGINGFACE_TOKEN`. Do not commit live tokens into notebooks.


In [ ]:
import math
import os
import random
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext
from pathlib import Path

CPU_CORES = max(1, os.cpu_count() or 1)
os.environ.setdefault("OMP_NUM_THREADS", str(CPU_CORES))
os.environ.setdefault("MKL_NUM_THREADS", str(CPU_CORES))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

import numpy as np
import torch
from datasets import load_dataset
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    InfNanRemoveLogitsProcessor,
    LogitsProcessorList,
)

try:
    from peft import LoraConfig, TaskType, get_peft_model
    PEFT_AVAILABLE = True
except Exception as exc:
    PEFT_AVAILABLE = False
    PEFT_IMPORT_ERROR = exc

# -----------------------------------------------------------------------------
# Paths and models
# -----------------------------------------------------------------------------
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("No HF token found in the environment. Public models will still load.")

GEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CLASSIFIER_PATH = "/mnt/main/prompt_classifier"
PREF_MODEL = "OpenAssistant/reward-model-deberta-v3-large-v2"
DATA_PATH = Path("prompts.jsonl")
OUTPUT_DIR = Path("two_head_attention_grpo_lora")

# -----------------------------------------------------------------------------
# Main controls
# -----------------------------------------------------------------------------
SEED = 42
DO_TRAIN = True
MAX_TRAIN_STEPS = None
DATASET_FRACTION = 0.10
TRAIN_FRACTION = 0.90
GROUP_SIZE = 5
GRPO_BASELINE_GROUP_SIZE = GROUP_SIZE
SAVE_EVERY = 100
PRINT_EVERY = 1
SUMMARY_EVERY = 10

# -----------------------------------------------------------------------------
# Generation and scoring
# -----------------------------------------------------------------------------
MAX_PROMPT_TOKENS = 1024
TRAIN_MAX_TOKENS = 1536
PREF_MAX_TOKENS = 512
REWARD_BATCH_SIZE = max(8, GROUP_SIZE + GRPO_BASELINE_GROUP_SIZE)
GENERATION_BATCH_SIZE = max(GROUP_SIZE, GRPO_BASELINE_GROUP_SIZE)
MAX_PARALLEL_WORKERS = CPU_CORES
MIN_RESPONSE_TOKENS = 4

# -----------------------------------------------------------------------------
# LoRA policy update
# -----------------------------------------------------------------------------
USE_LORA = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LR = 5e-5
WEIGHT_DECAY = 0.0
EPS = 0.20
KL_BETA = 0.03
GRAD_CLIP = 1.0
ATTENTION_TOPK_UPDATES = 2

# -----------------------------------------------------------------------------
# The only two custom attention heads
# -----------------------------------------------------------------------------
USE_TWO_ATTENTION_HEADS = True
ATTN_HEAD_DIM = 128
ATTN_NUM_HEADS = 4
ATTN_LR = 1e-4
ATTN_REWARD_TEMP = 0.75
ATTN_ENTROPY_LAMBDA = 0.005
UNIFORM_CANDIDATE_WEIGHT = 1.0 / GROUP_SIZE

NORMAL_MODALITY = 7
N_MODALITIES = 11

MODALITY_MAP = {
    0: "Concise",
    1: "Detailed",
    2: "Advanced",
    3: "Code_base",
    4: "Creative",
    5: "Formal",
    6: "Informal",
    7: "Normal",
    8: "Precise",
    9: "Reasoning",
    10: "Simple",
}

# Sampling profiles only. These are never inserted into the prompt text.
MODALITY_PROFILES = {
    0: dict(temperature=0.45, top_p=0.82, max_new_tokens=96, repetition_penalty=1.08),
    1: dict(temperature=0.70, top_p=0.90, max_new_tokens=192, repetition_penalty=1.05),
    2: dict(temperature=0.62, top_p=0.88, max_new_tokens=224, repetition_penalty=1.04),
    3: dict(temperature=0.35, top_p=0.82, max_new_tokens=256, repetition_penalty=1.03),
    4: dict(temperature=0.95, top_p=0.96, max_new_tokens=192, repetition_penalty=1.04),
    5: dict(temperature=0.50, top_p=0.86, max_new_tokens=160, repetition_penalty=1.06),
    6: dict(temperature=0.75, top_p=0.92, max_new_tokens=160, repetition_penalty=1.04),
    7: dict(temperature=0.65, top_p=0.90, max_new_tokens=160, repetition_penalty=1.05),
    8: dict(temperature=0.38, top_p=0.82, max_new_tokens=160, repetition_penalty=1.08),
    9: dict(temperature=0.55, top_p=0.88, max_new_tokens=256, repetition_penalty=1.04),
    10: dict(temperature=0.42, top_p=0.84, max_new_tokens=128, repetition_penalty=1.08),
}

SYSTEM_PROMPT = "You are a helpful assistant."


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print("Configuration ready.")


No HF token found in the environment. Public models will still load.
Configuration ready.


## 2. Load Models

The generator is trained with LoRA. Because the base model stays frozen, the notebook can compute a base-model anchor by temporarily disabling the adapter instead of loading a second reference model.


In [3]:
def module_device(module):
    try:
        return module.device
    except Exception:
        return next(module.parameters()).device


def to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}


device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print(f"Device: {device} | generator dtype: {dtype}")

if USE_LORA and not PEFT_AVAILABLE:
    raise RuntimeError(f"PEFT is required for USE_LORA=True: {PEFT_IMPORT_ERROR}")

gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL, token=HF_TOKEN)
if gen_tok.pad_token_id is None:
    gen_tok.pad_token = gen_tok.eos_token
gen_tok.padding_side = "left"

generator = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=dtype,
    device_map="auto",
    token=HF_TOKEN,
)

generator.config.use_cache = True

if USE_LORA:
    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    generator = get_peft_model(generator, lora_cfg)
    generator.print_trainable_parameters()
else:
    for p in generator.parameters():
        p.requires_grad_(True)

trainable_params = [p for p in generator.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)

base_config = generator.get_base_model().config if hasattr(generator, "get_base_model") else generator.config
GEN_HIDDEN = int(base_config.hidden_size)
print(f"Generator hidden size for attention heads: {GEN_HIDDEN}")

try:
    cls_tok = AutoTokenizer.from_pretrained(CLASSIFIER_PATH)
    classifier = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_PATH).to(device)
    classifier.eval()
    for p in classifier.parameters():
        p.requires_grad_(False)
    print("Prompt classifier loaded.")
except Exception as exc:
    classifier = None
    cls_tok = None
    print(f"Prompt classifier unavailable. Falling back to neutral routing. Reason: {exc}")

pref_tok = AutoTokenizer.from_pretrained(PREF_MODEL)
pref_mdl = AutoModelForSequenceClassification.from_pretrained(PREF_MODEL).to(device)
pref_mdl.eval()
for p in pref_mdl.parameters():
    p.requires_grad_(False)

print("Models ready.")


Device: cuda | generator dtype: torch.bfloat16


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/993 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Models ready.


## 3. Two Attention Heads

Only these two custom heads are used:

- `ClassifierPromptGenerationAttention`: mandatory before generation.
- `UserPreferenceUpdateAttention`: mandatory before the policy update.


In [4]:
class ClassifierPromptGenerationAttention(torch.nn.Module):
    # Attends over modality slots conditioned on classifier probabilities.
    def __init__(self, n_modalities=N_MODALITIES, dim=ATTN_HEAD_DIM, heads=ATTN_NUM_HEADS):
        super().__init__()
        self.mod_emb = torch.nn.Embedding(n_modalities, dim)
        self.prob_proj = torch.nn.Linear(1, dim)
        self.attn = torch.nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = torch.nn.LayerNorm(dim)
        self.score = torch.nn.Linear(dim, 1)
        self.gate_logit = torch.nn.Parameter(torch.tensor(-0.5))

    def forward(self, clf_probs):
        dev = self.mod_emb.weight.device
        probs = clf_probs.to(dev, dtype=torch.float32)
        probs = probs / probs.sum().clamp_min(1e-8)
        ids = torch.arange(probs.numel(), device=dev).unsqueeze(0)
        x = self.mod_emb(ids) + self.prob_proj(probs.view(1, -1, 1))
        y, attn_map = self.attn(x, x, x, need_weights=True, average_attn_weights=False)
        h = self.norm(x + y)
        learned = torch.softmax(self.score(h).squeeze(0).squeeze(-1), dim=-1)
        gate = torch.sigmoid(self.gate_logit)
        out = (1.0 - gate) * probs + gate * learned
        return out / out.sum().clamp_min(1e-8), gate, learned, attn_map


class UserPreferenceUpdateAttention(torch.nn.Module):
    # Attends over candidate responses using preference/reward features.
    def __init__(self, feature_dim, dim=ATTN_HEAD_DIM, heads=ATTN_NUM_HEADS):
        super().__init__()
        self.in_proj = torch.nn.Linear(feature_dim, dim)
        self.attn = torch.nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = torch.nn.LayerNorm(dim)
        self.score = torch.nn.Linear(dim, 1)
        self.gate_logit = torch.nn.Parameter(torch.tensor(-0.5))

    def forward(self, features, reward_prior):
        features = features.to(next(self.parameters()).device, dtype=torch.float32)
        reward_prior = reward_prior.to(features.device, dtype=torch.float32)
        reward_prior = reward_prior / reward_prior.sum().clamp_min(1e-8)
        x = self.in_proj(features).unsqueeze(0)
        y, attn_map = self.attn(x, x, x, need_weights=True, average_attn_weights=False)
        h = self.norm(x + y)
        learned = torch.softmax(self.score(h).squeeze(0).squeeze(-1), dim=-1)
        gate = torch.sigmoid(self.gate_logit)
        weights = (1.0 - gate) * reward_prior + gate * learned
        return weights / weights.sum().clamp_min(1e-8), gate, learned, attn_map


CANDIDATE_FEATURE_DIM = N_MODALITIES + 6
generation_head = preference_head = None
attention_optimizer = None

if USE_TWO_ATTENTION_HEADS:
    generation_head = ClassifierPromptGenerationAttention().to(device)
    preference_head = UserPreferenceUpdateAttention(CANDIDATE_FEATURE_DIM).to(device)
    attention_optimizer = torch.optim.AdamW(
        list(generation_head.parameters()) + list(preference_head.parameters()),
        lr=ATTN_LR,
    )
    print("Two attention heads ready:")
    print(f"  generation head params: {sum(p.numel() for p in generation_head.parameters()):,}")
    print(f"  preference head params: {sum(p.numel() for p in preference_head.parameters()):,}")
else:
    print("Attention heads disabled.")


Two attention heads ready:
  generation head params: 68,098
  preference head params: 68,738


## 4. Dataset

Expected input format: a JSONL file with a `prompt` field. The helper also accepts `text`, `instruction`, or `user` if needed.


In [5]:
def get_user_prompt(item):
    for key in ("prompt", "text", "instruction", "user"):
        if key in item and item[key] is not None and str(item[key]).strip():
            return str(item[key]).strip()
    raise KeyError(f"Could not find a prompt field in row keys: {list(item.keys())}")


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing {DATA_PATH}. Put prompts.jsonl next to this notebook or change DATA_PATH."
    )

full_ds = load_dataset("json", data_files=str(DATA_PATH))["train"]
total_all = len(full_ds)
n_use = max(1, int(total_all * DATASET_FRACTION))
dataset = full_ds.shuffle(seed=SEED).select(range(n_use))

split_idx = max(1, int(len(dataset) * TRAIN_FRACTION))
train_ds = dataset.select(range(split_idx))
val_ds = dataset.select(range(split_idx, len(dataset))) if split_idx < len(dataset) else dataset.select([])

print(f"Dataset: {len(train_ds)} train / {len(val_ds)} validation from {total_all} total rows")
print("Sample prompt:")
print(get_user_prompt(train_ds[0])[:500])


Generating train split: 0 examples [00:00, ? examples/s]

Dataset: 97 train / 11 validation from 1085 total rows
Sample prompt:
If we fold a paper and then apply pressure on the newly formed crease, it seems that the paper's surface gets a permanent deformation but what exactly has happened to the paper at a molecular scale?


## 5. Prompt Builder and Generation

This is the core anti-leak rule: `build_model_prompt` never receives a modality name, style instruction, rewritten prompt, or soft prefix.


In [6]:
def build_model_prompt(user_prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    if getattr(gen_tok, "chat_template", None):
        return gen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{user_prompt}\n<|assistant|>\n"


def set_generation_seed(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def is_cuda_oom(exc):
    return isinstance(exc, RuntimeError) and "out of memory" in str(exc).lower()


@torch.inference_mode()
def generate_response(user_prompt, profile, seed=None):
    return generate_responses_batch(user_prompt, profile, count=1, seed=seed)[0]


@torch.inference_mode()
def generate_responses_batch(user_prompt, profile, count=1, seed=None, batch_size=None):
    count = int(count)
    if count <= 0:
        return []
    if seed is not None:
        set_generation_seed(seed)

    batch_size = count if batch_size is None else max(1, min(int(batch_size), count))
    was_training = generator.training
    generator.eval()
    prompt_text = build_model_prompt(user_prompt)
    gen_device = module_device(generator)
    responses = []

    try:
        start = 0
        while start < count:
            current_batch = min(batch_size, count - start)
            try:
                inputs = gen_tok(
                    [prompt_text] * current_batch,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=MAX_PROMPT_TOKENS,
                )
                inputs = to_device(inputs, gen_device)

                out = generator.generate(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    do_sample=True,
                    temperature=float(profile.get("temperature", 0.65)),
                    top_p=float(profile.get("top_p", 0.90)),
                    repetition_penalty=float(profile.get("repetition_penalty", 1.05)),
                    max_new_tokens=int(profile.get("max_new_tokens", 160)),
                    pad_token_id=gen_tok.pad_token_id,
                    eos_token_id=gen_tok.eos_token_id,
                    logits_processor=LogitsProcessorList([InfNanRemoveLogitsProcessor()]),
                )
            except RuntimeError as exc:
                if torch.cuda.is_available() and is_cuda_oom(exc) and batch_size > 1:
                    torch.cuda.empty_cache()
                    batch_size = max(1, batch_size // 2)
                    print(f"CUDA OOM during generation; retrying with generation batch={batch_size}")
                    continue
                raise

            prompt_len = inputs["input_ids"].shape[1]
            for row in range(out.shape[0]):
                new_ids = out[row, prompt_len:]
                responses.append(gen_tok.decode(new_ids, skip_special_tokens=True).strip())
            start += current_batch
    finally:
        if was_training:
            generator.train()

    return responses


print("Prompt and generation helpers ready.")


Prompt and generation helpers ready.


## 6. Non-Invasive Modality Router

The router chooses which sampling profiles to try. It never edits the user prompt and never injects modality tokens or hidden prefixes into the generator.


In [7]:
def classifier_distribution(user_prompt):
    probs = np.zeros(N_MODALITIES, dtype=np.float32)
    if classifier is None or cls_tok is None:
        probs[NORMAL_MODALITY] = 1.0
        return probs

    cls_device = module_device(classifier)
    inputs = cls_tok(user_prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = to_device(inputs, cls_device)
    with torch.inference_mode():
        logits = classifier(**inputs).logits.float()[0]
        p = torch.softmax(logits, dim=-1).detach().cpu().numpy().astype(np.float32)

    upto = min(N_MODALITIES, len(p))
    probs[:upto] = p[:upto]
    if probs.sum() <= 0:
        probs[NORMAL_MODALITY] = 1.0
    else:
        probs = probs / probs.sum()
    return probs


def generation_attention_distribution(clf_probs):
    if not (USE_TWO_ATTENTION_HEADS and generation_head is not None):
        return clf_probs / (clf_probs.sum() + 1e-8), 0.0
    with torch.no_grad():
        dist, gate, _, _ = generation_head(torch.tensor(clf_probs, device=device))
    return dist.detach().cpu().numpy().astype(np.float32), float(gate.detach().cpu())


def select_modalities(user_prompt, k=GROUP_SIZE):
    clf_probs = classifier_distribution(user_prompt)
    gen_scores, gen_gate = generation_attention_distribution(clf_probs)
    ranked = list(np.argsort(gen_scores)[::-1])
    chosen = []
    for mid in ranked:
        if int(mid) not in chosen:
            chosen.append(int(mid))
        if len(chosen) == k:
            break
    return chosen, clf_probs, gen_scores, gen_gate


print("Generation router ready.")


Generation router ready.


## 7. Reward Model and Guard Rails

The reward model remains the main signal. Guard penalties are intentionally small and only catch obvious failure cases such as empty output, prompt leakage, and severe repetition.


In [8]:
def preference_rewards(user_prompt, responses):
    if not responses:
        return []

    texts = [user_prompt.strip() + "\n\n" + (r or "").strip() for r in responses]
    scores = []
    pref_device = module_device(pref_mdl)

    batch_size = max(1, min(REWARD_BATCH_SIZE, len(texts)))
    start = 0
    while start < len(texts):
        batch_texts = texts[start : start + batch_size]
        try:
            inputs = pref_tok(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=PREF_MAX_TOKENS,
            )
            inputs = to_device(inputs, pref_device)
            with torch.inference_mode():
                logits = pref_mdl(**inputs).logits.float()
                if logits.ndim == 2 and logits.shape[-1] > 1:
                    vals = logits[:, -1]
                else:
                    vals = logits.view(-1)
        except RuntimeError as exc:
            if torch.cuda.is_available() and is_cuda_oom(exc) and batch_size > 1:
                torch.cuda.empty_cache()
                batch_size = max(1, batch_size // 2)
                print(f"CUDA OOM during reward scoring; retrying with reward batch={batch_size}")
                continue
            raise

        scores.extend(vals.detach().cpu().tolist())
        start += len(batch_texts)

    return [float(s) for s in scores]


def guard_penalty(user_prompt, response):
    response = (response or "").strip()
    words = response.split()
    penalty = 0.0

    if not response:
        penalty -= 4.0
    if len(words) < MIN_RESPONSE_TOKENS:
        penalty -= 2.0

    prompt_head = user_prompt.strip().lower()[:80]
    if prompt_head and prompt_head in response.lower():
        penalty -= 1.0

    lines = [line.strip().lower() for line in response.splitlines() if line.strip()]
    if len(lines) >= 4:
        repeated_lines = len(lines) - len(set(lines))
        penalty -= min(1.5, 0.25 * repeated_lines)

    if len(words) >= 30:
        distinct_ratio = len(set(w.lower() for w in words)) / max(1, len(words))
        if distinct_ratio < 0.35:
            penalty -= min(2.0, (0.35 - distinct_ratio) * 6.0)

    return float(penalty)


def score_responses(user_prompt, responses):
    raw = preference_rewards(user_prompt, responses)
    if len(responses) > 1 and MAX_PARALLEL_WORKERS > 1:
        workers = min(MAX_PARALLEL_WORKERS, len(responses))
        with ThreadPoolExecutor(max_workers=workers) as pool:
            guards = list(pool.map(lambda r: guard_penalty(user_prompt, r), responses))
    else:
        guards = [guard_penalty(user_prompt, r) for r in responses]
    shaped = [r + g for r, g in zip(raw, guards)]
    return raw, shaped, guards


def normalize_advantages(values):
    arr = np.asarray(values, dtype=np.float32)
    if arr.size == 0:
        return arr
    if arr.size == 1 or float(arr.std()) < 1e-6:
        out = np.zeros_like(arr)
        out[int(np.argmax(arr))] = 1.0
        return out
    lo, hi = np.percentile(arr, [5, 95])
    arr = np.clip(arr, lo, hi)
    return (arr - arr.mean()) / (arr.std() + 1e-6)


print("Reward helpers ready.")


Reward helpers ready.


## 8. Response-Only Policy Update

Only response tokens receive gradient. Prompt tokens are masked with `-100`, so the model is not trained to imitate a modified or generated prompt.


In [9]:
def adapter_disabled():
    if USE_LORA and hasattr(generator, "disable_adapter"):
        return generator.disable_adapter()
    return nullcontext()


def trainable_l2_norm(params):
    total = 0.0
    with torch.no_grad():
        for p in params:
            total += float(p.detach().float().pow(2).sum().cpu())
    return math.sqrt(total)


def build_train_tensors(user_prompt, response):
    prompt_text = build_model_prompt(user_prompt)
    prompt_ids = gen_tok(prompt_text, add_special_tokens=False).input_ids
    response_ids = gen_tok((response or "").strip(), add_special_tokens=False).input_ids
    if gen_tok.eos_token_id is not None:
        response_ids = response_ids + [gen_tok.eos_token_id]
    if not response_ids:
        response_ids = [gen_tok.eos_token_id or gen_tok.pad_token_id]

    max_total = min(TRAIN_MAX_TOKENS, getattr(base_config, "max_position_embeddings", TRAIN_MAX_TOKENS))
    if len(response_ids) >= max_total:
        response_ids = response_ids[: max_total - 1]
    keep_prompt = max(1, max_total - len(response_ids))
    if len(prompt_ids) > keep_prompt:
        prompt_ids = prompt_ids[-keep_prompt:]

    input_ids = prompt_ids + response_ids
    labels = [-100] * len(prompt_ids) + response_ids
    attention_mask = [1] * len(input_ids)
    dev = module_device(generator)
    return (
        torch.tensor([input_ids], dtype=torch.long, device=dev),
        torch.tensor([attention_mask], dtype=torch.long, device=dev),
        torch.tensor([labels], dtype=torch.long, device=dev),
    )


def response_mean_logprob(user_prompt, response, disable_adapter=False):
    input_ids, attention_mask, labels = build_train_tensors(user_prompt, response)
    ctx = adapter_disabled() if disable_adapter else nullcontext()
    with ctx:
        out = generator(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)

    logits = out.logits[:, :-1, :].float()
    target_ids = input_ids[:, 1:]
    valid = labels[:, 1:] != -100
    token_logp = torch.log_softmax(logits, dim=-1).gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)
    denom = valid.sum().clamp_min(1)
    return (token_logp * valid).sum() / denom


def normalize_advantages(values):
    arr = np.asarray(values, dtype=np.float32)
    if arr.size <= 1 or float(arr.std()) < 1e-6:
        out = np.zeros_like(arr)
        out[int(np.argmax(arr))] = 1.0
        return out
    lo, hi = np.percentile(arr, [5, 95])
    arr = np.clip(arr, lo, hi)
    return (arr - arr.mean()) / (arr.std() + 1e-6)


def select_policy_indices(shaped_rewards, preference_weights):
    rewards = np.asarray(shaped_rewards, dtype=np.float32)
    pref = np.asarray(preference_weights, dtype=np.float32)
    pref = pref / (pref.sum() + 1e-8)
    best_idx = int(np.argmax(rewards))
    positive_adv = np.maximum(normalize_advantages(rewards), 0.0)
    if float(positive_adv.max()) <= 1e-8:
        positive_adv[best_idx] = 1.0
    gains = positive_adv * np.maximum(0.10, pref / (pref.mean() + 1e-8))
    gains[best_idx] = max(gains[best_idx], 1.0)
    gains = gains / (gains.max() + 1e-8)
    ranked = list(np.argsort(gains)[::-1])
    train_indices = [int(i) for i in ranked if gains[i] > 1e-8][: max(1, ATTENTION_TOPK_UPDATES)]
    return train_indices, gains.astype(np.float32), best_idx


def estimate_group_loss(shaped_rewards, preference_weights=None):
    rewards = np.asarray(shaped_rewards, dtype=np.float32)
    if preference_weights is None:
        preference_weights = np.ones_like(rewards, dtype=np.float32) / max(1, len(rewards))
    train_indices, gains, _ = select_policy_indices(rewards, preference_weights)
    return -float(np.mean([gains[i] for i in train_indices])) if train_indices else 0.0


def update_policy(user_prompt, responses, shaped_rewards, preference_weights):
    if not responses:
        return {"loss": 0.0, "trained": 0, "best_idx": None, "grad_norm": 0.0, "adapter_delta": 0.0}

    train_indices, advantages, best_idx = select_policy_indices(shaped_rewards, preference_weights)
    old_logps = {}
    base_logps = {}

    generator.eval()
    with torch.no_grad():
        for idx in train_indices:
            old_logps[idx] = response_mean_logprob(user_prompt, responses[idx]).detach()
            if USE_LORA:
                base_logps[idx] = response_mean_logprob(user_prompt, responses[idx], disable_adapter=True).detach()
            else:
                base_logps[idx] = old_logps[idx]

    generator.train()
    optimizer.zero_grad(set_to_none=True)
    losses = []
    for idx in train_indices:
        adv = float(advantages[idx])
        if abs(adv) < 1e-8:
            continue
        cur_logp = response_mean_logprob(user_prompt, responses[idx])
        old_logp = old_logps[idx].to(cur_logp.device)
        base_logp = base_logps[idx].to(cur_logp.device)
        ratio = torch.exp(torch.clamp(cur_logp - old_logp, min=-5.0, max=5.0))
        adv_t = torch.tensor(adv, dtype=torch.float32, device=cur_logp.device)
        unclipped = ratio * adv_t
        clipped = torch.clamp(ratio, 1.0 - EPS, 1.0 + EPS) * adv_t
        policy_loss = -torch.minimum(unclipped, clipped)
        base_anchor = (cur_logp - base_logp).pow(2)
        losses.append(policy_loss + KL_BETA * base_anchor)

    if not losses:
        return {"loss": 0.0, "trained": 0, "best_idx": best_idx, "grad_norm": 0.0, "adapter_delta": 0.0}

    norm_before = trainable_l2_norm(trainable_params)
    total_loss = torch.stack(losses).mean()
    total_loss.backward()
    grad_norm_t = torch.nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP)
    optimizer.step()
    norm_after = trainable_l2_norm(trainable_params)
    return {
        "loss": float(total_loss.detach().cpu()),
        "trained": len(losses),
        "best_idx": best_idx,
        "grad_norm": float(grad_norm_t.detach().float().cpu()),
        "adapter_delta": norm_after - norm_before,
    }


print("Policy update helpers ready.")


Policy update helpers ready.


## 8. Attention-Head Training Helpers

The heads train on auxiliary signals from the same RL batch. This lets the heads become useful before they are trusted with much influence.


In [10]:
def zscore_np(values):
    arr = np.asarray(values, dtype=np.float32)
    if arr.size == 0 or float(arr.std()) < 1e-6:
        return np.zeros_like(arr, dtype=np.float32)
    return ((arr - arr.mean()) / (arr.std() + 1e-6)).astype(np.float32)


def preference_target_distribution(shaped_rewards):
    rewards = torch.tensor(shaped_rewards, dtype=torch.float32, device=device)
    if rewards.numel() == 1:
        return torch.ones_like(rewards)
    rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-6)
    return torch.softmax(rewards / ATTN_REWARD_TEMP, dim=0)


def build_candidate_features(mod_ids, clf_probs, gen_scores, raw_rewards, shaped_rewards, guard_penalties, responses):
    raw_z = zscore_np(raw_rewards)
    shaped_z = zscore_np(shaped_rewards)
    guards = np.asarray(guard_penalties, dtype=np.float32)
    lengths = np.asarray([len((r or "").split()) for r in responses], dtype=np.float32)
    lengths = np.log1p(lengths) / np.log1p(max(float(lengths.max()), 1.0))

    rows = []
    for i, mid in enumerate(mod_ids):
        one_hot = np.zeros(N_MODALITIES, dtype=np.float32)
        one_hot[int(mid)] = 1.0
        extra = np.array([
            float(clf_probs[int(mid)]),
            float(gen_scores[int(mid)]),
            float(raw_z[i]),
            float(shaped_z[i]),
            float(guards[i]),
            float(lengths[i]),
        ], dtype=np.float32)
        rows.append(np.concatenate([one_hot, extra], axis=0))
    return torch.tensor(np.stack(rows), dtype=torch.float32, device=device)


def preference_attention_distribution(candidate_features, shaped_rewards):
    target = preference_target_distribution(shaped_rewards)
    if not (USE_TWO_ATTENTION_HEADS and preference_head is not None):
        return target.detach().cpu().numpy().astype(np.float32), 0.0
    with torch.no_grad():
        weights, gate, _, _ = preference_head(candidate_features, target)
    return weights.detach().cpu().numpy().astype(np.float32), float(gate.detach().cpu())


def train_attention_heads(mod_ids, clf_probs, gen_scores, raw_rewards, shaped_rewards, guard_penalties, responses):
    candidate_features = build_candidate_features(
        mod_ids, clf_probs, gen_scores, raw_rewards, shaped_rewards, guard_penalties, responses
    )
    target = preference_target_distribution(shaped_rewards).detach()
    info = {"gen_loss": 0.0, "pref_loss": 0.0, "gen_gate": 0.0, "pref_gate": 0.0}

    if not (USE_TWO_ATTENTION_HEADS and attention_optimizer is not None):
        weights = target.detach().cpu().numpy().astype(np.float32)
        return candidate_features, weights, info

    attention_optimizer.zero_grad(set_to_none=True)
    losses = []

    gen_dist, gen_gate, _, _ = generation_head(torch.tensor(clf_probs, dtype=torch.float32, device=device))
    idx_t = torch.tensor(mod_ids, dtype=torch.long, device=device)
    selected_gen = gen_dist[idx_t]
    selected_gen = selected_gen / selected_gen.sum().clamp_min(1e-8)
    gen_loss = -(target * torch.log(selected_gen + 1e-8)).sum()
    losses.append(gen_loss)
    info["gen_loss"] = float(gen_loss.detach().cpu())
    info["gen_gate"] = float(gen_gate.detach().cpu())

    pref_weights, pref_gate, _, _ = preference_head(candidate_features, target)
    pref_entropy = -(pref_weights * torch.log(pref_weights + 1e-8)).sum()
    pref_loss = -(target * torch.log(pref_weights + 1e-8)).sum() - ATTN_ENTROPY_LAMBDA * pref_entropy
    losses.append(pref_loss)
    info["pref_loss"] = float(pref_loss.detach().cpu())
    info["pref_gate"] = float(pref_gate.detach().cpu())

    total = torch.stack(losses).mean()
    total.backward()
    torch.nn.utils.clip_grad_norm_(
        list(generation_head.parameters()) + list(preference_head.parameters()),
        GRAD_CLIP,
    )
    attention_optimizer.step()

    weights, pref_gate_after = preference_attention_distribution(candidate_features, shaped_rewards)
    info["pref_gate"] = pref_gate_after
    with torch.no_grad():
        _, gen_gate_after, _, _ = generation_head(torch.tensor(clf_probs, dtype=torch.float32, device=device))
    info["gen_gate"] = float(gen_gate_after.detach().cpu())
    return candidate_features, weights, info


print("Two-head attention training helpers ready.")


Two-head attention training helpers ready.


## 9. Tracking and Reporting


In [11]:
history = []
per_mod = defaultdict(lambda: {"novel": [], "grpo": [], "count": 0})


def avg(values):
    values = [v for v in values if v is not None and not np.isnan(v)]
    return float(np.mean(values)) if values else 0.0


def paired_stats(window=None):
    rows = [h for h in history if not np.isnan(h["grpo_reward"])]
    if window is not None:
        rows = rows[-int(window):]
    if not rows:
        return {"n": 0, "avg_delta": 0.0, "win_rate": 0.0}
    deltas = [h["novel_reward"] - h["grpo_reward"] for h in rows]
    wins = [h["novel_reward"] > h["grpo_reward"] for h in rows]
    return {
        "n": len(rows),
        "avg_delta": float(np.mean(deltas)),
        "win_rate": 100.0 * float(np.mean(wins)),
    }


def save_checkpoint(tag="latest"):
    out_dir = OUTPUT_DIR / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    generator.save_pretrained(out_dir)
    gen_tok.save_pretrained(out_dir)
    return out_dir


def format_step_report(step, total, rec):
    delta = rec["novel_reward"] - rec["grpo_reward"]
    s = paired_stats()
    s10 = paired_stats(window=10)
    pref_signal = rec["best_preference_weight"] - UNIFORM_CANDIDATE_WEIGHT
    return (
        f"{step:>4}/{total} | "
        f"GRPO R={rec['grpo_reward']:+.3f} L={rec['grpo_loss']:+.3f} | "
        f"NOVEL R={rec['novel_reward']:+.3f} L={rec['novel_loss']:+.3f} mod={rec['best_modality']:<10} | "
        f"dR={delta:+.3f} avgdR={s['avg_delta']:+.3f} WR={s['win_rate']:>5.1f}% last10={s10['avg_delta']:+.3f} | "
        f"gen_gate={rec['gen_gate']:.3f} pref_w={rec['best_preference_weight']:.3f}({pref_signal:+.3f}) "
        f"pref_gate={rec['pref_gate']:.3f} dW={rec['adapter_delta']:+.1e}"
    )


def print_final_report():
    if not history:
        print("No training history yet.")
        return
    grpo = [h["grpo_reward"] for h in history]
    novel = [h["novel_reward"] for h in history]
    s = paired_stats()
    print("\n" + "=" * 88)
    print(f"Two-head attention GRPO final report | steps={len(history)}")
    print("=" * 88)
    print(f"Mean GRPO reward       : {avg(grpo):+.4f}")
    print(f"Mean NOVEL reward      : {avg(novel):+.4f}")
    print(f"Mean delta NOVEL-GRPO  : {s['avg_delta']:+.4f}")
    print(f"NOVEL win rate vs GRPO : {s['win_rate']:.1f}%")
    print("-" * 88)
    for mid in sorted(MODALITY_MAP):
        name = MODALITY_MAP[mid]
        vals = per_mod[name]["novel"]
        if vals:
            print(f"  {name:<10} n={len(vals):>4} novel_avg={avg(vals):+.4f}")
    print("=" * 88 + "\n")


print("Tracking helpers ready.")


Tracking helpers ready.


## 10. Training Loop

Each step prints a minimal GRPO-vs-NOVEL comparison:

- `GRPO`: best of `G` normal-profile generations.
- `NOVEL`: best of `G` classifier-attention generations, updated with preference attention.


In [12]:
_PARALLEL_RUNTIME_CONFIGURED = False


def configure_parallel_runtime():
    global _PARALLEL_RUNTIME_CONFIGURED
    if _PARALLEL_RUNTIME_CONFIGURED:
        return

    os.environ["OMP_NUM_THREADS"] = str(CPU_CORES)
    os.environ["MKL_NUM_THREADS"] = str(CPU_CORES)
    os.environ["TOKENIZERS_PARALLELISM"] = "true"

    try:
        torch.set_num_threads(CPU_CORES)
    except RuntimeError as exc:
        print(f"CPU intra-op threads already locked by PyTorch: {exc}")

    try:
        torch.set_num_interop_threads(max(1, min(CPU_CORES, 8)))
    except RuntimeError as exc:
        print(f"CPU inter-op threads already locked by PyTorch: {exc}")

    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
        try:
            for cuda_idx in range(torch.cuda.device_count()):
                torch.cuda.set_per_process_memory_fraction(1.0, device=cuda_idx)
        except Exception as exc:
            print(f"CUDA memory fraction already fixed by runtime: {exc}")
        torch.cuda.empty_cache()

    _PARALLEL_RUNTIME_CONFIGURED = True


def run_training(max_steps=MAX_TRAIN_STEPS):
    total = len(train_ds) if max_steps is None else min(int(max_steps), len(train_ds))
    configure_parallel_runtime()

    print("\n" + "=" * 88)
    print("Two-head attention GRPO training")
    print(f"comparison: GRPO(G={GRPO_BASELINE_GROUP_SIZE}) vs NOVEL(G={GROUP_SIZE}) | LoRA={USE_LORA}")
    print(f"parallel: CPU cores={CPU_CORES}, generation batch={GENERATION_BATCH_SIZE}, reward batch={REWARD_BATCH_SIZE}")
    if torch.cuda.is_available():
        cuda_mem = []
        for cuda_idx in range(torch.cuda.device_count()):
            free_mem, total_mem = torch.cuda.mem_get_info(cuda_idx)
            cuda_mem.append(f"{cuda_idx}:{free_mem / 2**30:.1f}/{total_mem / 2**30:.1f} GiB")
        print("cuda memory free/total: " + ", ".join(cuda_mem))
    print("NOVEL uses exactly two heads: generation attention + preference update attention")
    print("=" * 88)
    print("step | GRPO reward/loss | NOVEL reward/loss | reward delta/running win rate | attention state\n")

    start_time = time.time()
    try:
        for step in range(1, total + 1):
            user_prompt = get_user_prompt(train_ds[step - 1])

            # NOVEL: generation candidates chosen by classifier-probability attention.
            mod_ids, clf_probs, gen_scores, gen_gate_pre = select_modalities(user_prompt, GROUP_SIZE)
            novel_responses = []
            for rank, mid in enumerate(mod_ids):
                novel_responses.extend(generate_responses_batch(
                    user_prompt,
                    MODALITY_PROFILES[mid],
                    count=1,
                    seed=SEED + step * 100 + rank,
                    batch_size=1,
                ))

            # GRPO baseline: same model state, same prompt, G normal-profile samples.
            grpo_responses = generate_responses_batch(
                user_prompt,
                MODALITY_PROFILES[NORMAL_MODALITY],
                count=GRPO_BASELINE_GROUP_SIZE,
                seed=SEED + step * 1000 + 99,
                batch_size=GENERATION_BATCH_SIZE,
            )

            combined_responses = novel_responses + grpo_responses
            combined_raw, combined_shaped, combined_guards = score_responses(user_prompt, combined_responses)
            split = len(novel_responses)
            novel_raw = combined_raw[:split]
            novel_shaped = combined_shaped[:split]
            novel_guards = combined_guards[:split]
            grpo_raw = combined_raw[split:]
            grpo_shaped = combined_shaped[split:]
            grpo_guards = combined_guards[split:]

            best_idx = int(np.argmax(novel_shaped))
            best_mid = mod_ids[best_idx]
            best_name = MODALITY_MAP[best_mid]
            grpo_best_idx = int(np.argmax(grpo_shaped))
            grpo_reward = float(grpo_raw[grpo_best_idx])
            grpo_loss = estimate_group_loss(grpo_shaped)

            # Train both attention heads from the observed preference rewards, then update LoRA.
            _, preference_weights, attn_info = train_attention_heads(
                mod_ids,
                clf_probs,
                gen_scores,
                novel_raw,
                novel_shaped,
                novel_guards,
                novel_responses,
            )
            update_info = update_policy(
                user_prompt,
                novel_responses,
                novel_shaped,
                preference_weights,
            )

            rec = {
                "step": step,
                "grpo_reward": grpo_reward,
                "grpo_loss": float(grpo_loss),
                "novel_reward": float(novel_raw[best_idx]),
                "novel_shaped_reward": float(novel_shaped[best_idx]),
                "novel_loss": float(update_info["loss"]),
                "best_modality": best_name,
                "best_modality_id": best_mid,
                "gen_loss": float(attn_info.get("gen_loss", 0.0)),
                "pref_loss": float(attn_info.get("pref_loss", 0.0)),
                "gen_gate": float(attn_info.get("gen_gate", gen_gate_pre)),
                "pref_gate": float(attn_info.get("pref_gate", 0.0)),
                "best_preference_weight": float(preference_weights[best_idx]),
                "trained": int(update_info["trained"]),
                "grad_norm": float(update_info.get("grad_norm", 0.0)),
                "adapter_delta": float(update_info.get("adapter_delta", 0.0)),
            }
            history.append(rec)
            per_mod[best_name]["novel"].append(rec["novel_reward"])
            per_mod[best_name]["grpo"].append(grpo_reward)
            per_mod[best_name]["count"] += 1

            if PRINT_EVERY and step % PRINT_EVERY == 0:
                print(format_step_report(step, total, rec))

            if SUMMARY_EVERY and step % SUMMARY_EVERY == 0:
                s = paired_stats(window=SUMMARY_EVERY)
                print(f"      last {SUMMARY_EVERY}: avgdR={s['avg_delta']:+.3f} WR={s['win_rate']:.1f}%")

            if SAVE_EVERY and step % SAVE_EVERY == 0:
                out_dir = save_checkpoint(f"step-{step}")
                print(f"      saved checkpoint: {out_dir}")

            if torch.cuda.is_available() and step % 5 == 0:
                torch.cuda.empty_cache()

    except KeyboardInterrupt:
        print("\nStopped by user. Printing partial results and saving latest adapter.")
    finally:
        elapsed = time.time() - start_time
        print(f"Elapsed: {elapsed / 60.0:.1f} minutes")
        out_dir = save_checkpoint("latest")
        print(f"latest adapter saved to: {out_dir}")
        print_final_report()


if DO_TRAIN:
    run_training(MAX_TRAIN_STEPS)
else:
    print("DO_TRAIN=False. Set it to True when you want to run training.")



Two-head attention GRPO training
comparison: GRPO(G=5) vs NOVEL(G=5) | LoRA=True
NOVEL uses exactly two heads: generation attention + preference update attention
step | GRPO reward/loss | NOVEL reward/loss | reward delta/running win rate | attention state

   1/97 | GRPO R=+0.995 L=-0.844 | NOVEL R=+1.817 L=-0.597 mod=Reasoning  | dR=+0.823 avgdR=+0.823 WR=100.0% last10=+0.823 | gen_gate=0.378 pref_w=0.409(+0.209) pref_gate=0.378 dW=+5.5e-04
   2/97 | GRPO R=-0.270 L=-0.965 | NOVEL R=-0.197 L=-0.692 mod=Precise    | dR=+0.073 avgdR=+0.448 WR=100.0% last10=+0.448 | gen_gate=0.378 pref_w=0.329(+0.129) pref_gate=0.377 dW=+1.9e-03
   3/97 | GRPO R=+0.681 L=-0.746 | NOVEL R=+0.710 L=-0.789 mod=Concise    | dR=+0.029 avgdR=+0.308 WR=100.0% last10=+0.308 | gen_gate=0.378 pref_w=0.346(+0.146) pref_gate=0.377 dW=+2.1e-03
   4/97 | GRPO R=+5.203 L=-0.567 | NOVEL R=+6.487 L=-0.506 mod=Reasoning  | dR=+1.284 avgdR=+0.552 WR=100.0% last10=+0.552 | gen_gate=0.378 pref_w=0.493(+0.293) pref_gate=0.37